# GHG Price Sensitivity with Fixed Baseline Diet

Explores how the food system responds to increasing GHG prices when diet is fixed
to baseline values. Since consumption cannot change, all adaptation happens on the
production side: crop switching, land use change, and livestock adjustments.

**Figure 1**: Net emissions by gas + per-gas source breakdown

**Figure 2**: LUC emissions and area by continent and land type

**Figure 3**: Objective cost breakdown

**Figure 4**: Feed breakdown by category and animal

In [ ]:
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
from sensitivity_utils import (
    ANIMAL_COLORS,
    CONTINENT_ORDER,
    FEED_COLORS,
    FEED_ORDER,
    FONTSIZE_TITLE,
    GAS_CMAPS,
    GAS_DISPLAY,
    LUC_TYPE_DISPLAY,
    PRETTY_NAMES_EMISSIONS,
    PRETTY_NAMES_GAS,
    extract_scenarios_with_param,
    get_log_ticks,
    load_emissions_by_source,
    load_feed_breakdown,
    load_luc_breakdown,
    load_net_emissions,
    load_net_emissions_by_gas,
    load_objective_from_analysis,
    plot_stacked_emissions,
    prepare_objective_data,
)

In [ ]:
PROJECT_ROOT = Path("..").resolve()
CONFIG_NAME = "ghg_sensitivity_fixed_diet"

## Extract scenarios

In [ ]:
ghg_scenarios_all = extract_scenarios_with_param(
    PROJECT_ROOT,
    CONFIG_NAME,
    param_path=["emissions", "ghg_price"],
    scenario_prefix="ghg_",
)
ghg_param_values_full = [p for p, _, _ in ghg_scenarios_all]
ghg_scenarios = [(p, s, f) for p, s, f in ghg_scenarios_all if f.exists()]
print(f"GHG: {len(ghg_scenarios)}/{len(ghg_scenarios_all)} solved")
print(f"GHG prices: {[p for p, _, _ in ghg_scenarios]}")

In [ ]:
GHG_XTICKS, GHG_XTICKLABELS = get_log_ticks(ghg_param_values_full)
GHG_XLABEL = "GHG price [USD/tCO\u2082eq]"

## Load data

In [ ]:
# Net emissions by gas
df_gas = load_net_emissions_by_gas(
    ghg_scenarios, PROJECT_ROOT, CONFIG_NAME, "ghg_price"
)
print(f"By-gas shape: {df_gas.shape}")
print(df_gas)

# Net emissions total
net_emissions = load_net_emissions(
    ghg_scenarios, PROJECT_ROOT, CONFIG_NAME, "ghg_price"
)
print(
    f"\nNet emissions range: {net_emissions.min():.2f} to {net_emissions.max():.2f} GtCO2eq"
)

In [ ]:
# Per-source breakdown
by_source = load_emissions_by_source(
    ghg_scenarios, PROJECT_ROOT, CONFIG_NAME, "ghg_price"
)
for gas in ["CO2", "CH4", "N2O"]:
    cols = list(by_source[gas].columns) if not by_source[gas].empty else []
    print(f"{gas}: {cols}")

In [ ]:
# LUC breakdowns
luc_continent = load_luc_breakdown(
    ghg_scenarios, PROJECT_ROOT, CONFIG_NAME, "ghg_price", groupby="continent"
)
luc_type = load_luc_breakdown(
    ghg_scenarios, PROJECT_ROOT, CONFIG_NAME, "ghg_price", groupby="land_type"
)
luc_continent_area = load_luc_breakdown(
    ghg_scenarios,
    PROJECT_ROOT,
    CONFIG_NAME,
    "ghg_price",
    groupby="continent",
    quantity="area",
)
luc_type_area = load_luc_breakdown(
    ghg_scenarios,
    PROJECT_ROOT,
    CONFIG_NAME,
    "ghg_price",
    groupby="land_type",
    quantity="area",
)
print(f"LUC continent cols: {list(luc_continent.columns)}")
print(f"LUC type cols: {list(luc_type.columns)}")

In [ ]:
# Objective breakdown
df_obj = load_objective_from_analysis(
    ghg_scenarios, PROJECT_ROOT, CONFIG_NAME, "ghg_price"
)
df_obj = prepare_objective_data(df_obj)
print(f"Objective shape: {df_obj.shape}")
print(f"Categories: {df_obj.columns.tolist()}")
print(df_obj)

In [ ]:
# Feed breakdowns
feed_cat = load_feed_breakdown(
    ghg_scenarios, PROJECT_ROOT, CONFIG_NAME, "ghg_price", groupby="feed_category"
)
feed_animal = load_feed_breakdown(
    ghg_scenarios, PROJECT_ROOT, CONFIG_NAME, "ghg_price", groupby="animal"
)
print(f"Feed by category: {list(feed_cat.columns)}")
print(f"Feed by animal: {list(feed_animal.columns)}")

## Figure 1: Emissions overview

In [ ]:
# Helper functions
def assign_gas_colors(df):
    gas_colors = {
        GAS_DISPLAY["co2"]: "#636363",
        GAS_DISPLAY["ch4"]: "#31a354",
        GAS_DISPLAY["n2o"]: "#e6550d",
    }
    return {col: gas_colors.get(col, "#999999") for col in df.columns}


def assign_source_colors_for_gas(gas, all_sources):
    cmap = matplotlib.colormaps[GAS_CMAPS.get(gas, "Blues")]
    n = len(all_sources)
    return {
        src: cmap(0.3 + 0.55 * i / max(n - 1, 1)) for i, src in enumerate(all_sources)
    }


def prepare_source_df(df, threshold=0.005):
    if df.empty:
        return df
    significant = df.columns[df.abs().max() >= threshold]
    df = df[significant]
    pos_cols = sorted(
        [c for c in df.columns if df[c].mean() >= 0],
        key=lambda c: df[c].mean(),
        reverse=True,
    )
    neg_cols = sorted(
        [c for c in df.columns if df[c].mean() < 0], key=lambda c: df[c].mean()
    )
    return df[pos_cols + neg_cols]

In [ ]:
gas_colors = assign_gas_colors(df_gas)
gas_order = df_gas.mean().sort_values(ascending=False).index.tolist()
df_gas_plot = df_gas[gas_order]

# Prepare source DataFrames
source_dfs = {}
source_colors = {}
for gas in ["CO2", "CH4", "N2O"]:
    df_src = prepare_source_df(by_source[gas])
    source_dfs[gas] = df_src
    source_colors[gas] = assign_source_colors_for_gas(
        gas, sorted(df_src.columns.tolist())
    )

# Build figure: 4 rows x 1 column
fig, axes = plt.subplots(4, 1, figsize=(5, 8))

# Row 0: Net emissions by gas
plot_stacked_emissions(
    df_gas_plot,
    gas_colors,
    axes[0],
    xlabel="",
    ylabel="Net emissions [GtCO\u2082eq]",
    panel_label="a",
    x_ticks=GHG_XTICKS,
    x_ticklabels=GHG_XTICKLABELS,
    min_height_for_label=0.5,
    pretty_names=PRETTY_NAMES_GAS,
)

# Rows 1-3: Per-gas source breakdown
for row, gas in enumerate(["CO2", "CH4", "N2O"], 1):
    df_src = source_dfs[gas]
    colors = {
        src: source_colors[gas][src]
        for src in df_src.columns
        if src in source_colors[gas]
    }
    xlabel = GHG_XLABEL if row == 3 else ""
    plot_stacked_emissions(
        df_src,
        colors,
        axes[row],
        xlabel=xlabel,
        ylabel=f"{GAS_DISPLAY[gas]} emissions [GtCO\u2082eq]",
        panel_label="bcd"[row - 1],
        x_ticks=GHG_XTICKS,
        x_ticklabels=GHG_XTICKLABELS,
        min_height_for_label=0.08,
        pretty_names=PRETTY_NAMES_EMISSIONS,
    )

# Remove x-labels from upper rows
for row in range(3):
    axes[row].set_xticklabels([])

axes[0].set_title(
    "GHG price sensitivity (fixed diet)",
    fontsize=FONTSIZE_TITLE + 2,
    fontweight="bold",
    pad=10,
)
fig.align_ylabels(axes)
plt.tight_layout()
plt.show()

## Figure 2: Land use change

In [ ]:
CONTINENT_COLORS = {
    "Africa": "#e41a1c",
    "Americas": "#377eb8",
    "Asia": "#4daf4a",
    "Europe": "#984ea3",
    "Oceania": "#ff7f00",
    "Other": "#999999",
}
LUC_TYPE_COLORS = {
    "Cropland expansion": "#d62728",
    "Pasture expansion": "#ff7f0e",
    "Cropland sparing": "#1f77b4",
    "Grassland sparing (convertible)": "#2ca02c",
    "Grassland sparing (marginal)": "#98df8a",
}


def prepare_luc_df(df, col_order=None, threshold=0.005):
    if df.empty:
        return df
    significant = df.columns[df.abs().max() >= threshold]
    df = df[significant]
    if col_order is not None:
        ordered = [c for c in col_order if c in df.columns]
        extra = [c for c in df.columns if c not in ordered]
        df = df[ordered + extra]
    return df


fig, axes = plt.subplots(2, 2, figsize=(8, 5))

# Top-left: LUC emissions by continent
df = prepare_luc_df(luc_continent, CONTINENT_ORDER)
colors = {c: CONTINENT_COLORS.get(c, "#999999") for c in df.columns}
plot_stacked_emissions(
    df,
    colors,
    axes[0, 0],
    xlabel="",
    ylabel="LUC emissions [GtCO\u2082]",
    panel_label="a",
    x_ticks=GHG_XTICKS,
    x_ticklabels=GHG_XTICKLABELS,
    min_height_for_label=0.3,
)

# Top-right: LUC emissions by type
df = prepare_luc_df(luc_type, list(LUC_TYPE_COLORS.keys()))
colors = {c: LUC_TYPE_COLORS.get(c, "#999999") for c in df.columns}
plot_stacked_emissions(
    df,
    colors,
    axes[0, 1],
    xlabel="",
    ylabel="",
    panel_label="b",
    x_ticks=GHG_XTICKS,
    x_ticklabels=GHG_XTICKLABELS,
    min_height_for_label=0.3,
    pretty_names=LUC_TYPE_DISPLAY,
)

# Bottom-left: LUC area by continent
df = prepare_luc_df(luc_continent_area, CONTINENT_ORDER)
colors = {c: CONTINENT_COLORS.get(c, "#999999") for c in df.columns}
plot_stacked_emissions(
    df,
    colors,
    axes[1, 0],
    xlabel=GHG_XLABEL,
    ylabel="Land use change [Mha]",
    panel_label="c",
    x_ticks=GHG_XTICKS,
    x_ticklabels=GHG_XTICKLABELS,
    min_height_for_label=30,
)

# Bottom-right: LUC area by type
df = prepare_luc_df(luc_type_area, list(LUC_TYPE_COLORS.keys()))
colors = {c: LUC_TYPE_COLORS.get(c, "#999999") for c in df.columns}
plot_stacked_emissions(
    df,
    colors,
    axes[1, 1],
    xlabel=GHG_XLABEL,
    ylabel="",
    panel_label="d",
    x_ticks=GHG_XTICKS,
    x_ticklabels=GHG_XTICKLABELS,
    min_height_for_label=30,
    pretty_names=LUC_TYPE_DISPLAY,
)

# Remove x-labels from top row
for col in range(2):
    axes[0, col].set_xticklabels([])

# Share y-axis within rows
for row in range(2):
    y_min = min(ax.get_ylim()[0] for ax in axes[row, :])
    y_max = max(ax.get_ylim()[1] for ax in axes[row, :])
    for col in range(2):
        axes[row, col].set_ylim(y_min, y_max)

axes[0, 0].set_title(
    "By continent", fontsize=FONTSIZE_TITLE + 2, fontweight="bold", pad=8
)
axes[0, 1].set_title(
    "By land type", fontsize=FONTSIZE_TITLE + 2, fontweight="bold", pad=8
)

fig.align_ylabels(axes[:, 0])
plt.tight_layout()
plt.show()

## Figure 3: Objective cost breakdown

In [ ]:
from sensitivity_utils import plot_objective_sensitivity

fig, ax = plt.subplots(1, 1, figsize=(6, 3.5))

plot_objective_sensitivity(
    df_obj,
    ax,
    xlabel=GHG_XLABEL,
    panel_label="",
    x_ticks=GHG_XTICKS,
    x_ticklabels=GHG_XTICKLABELS,
)
ax.set_title(
    "Objective cost breakdown (fixed diet)",
    fontsize=FONTSIZE_TITLE + 2,
    fontweight="bold",
    pad=10,
)

plt.tight_layout()
plt.show()

## Figure 4: Feed breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))

# Feed by category
df = prepare_luc_df(feed_cat, FEED_ORDER)
colors = {c: FEED_COLORS.get(c, "#999999") for c in df.columns}
plot_stacked_emissions(
    df,
    colors,
    axes[0],
    xlabel=GHG_XLABEL,
    ylabel="Feed use [Gt DM]",
    panel_label="a",
    x_ticks=GHG_XTICKS,
    x_ticklabels=GHG_XTICKLABELS,
    min_height_for_label=0.3,
)
axes[0].set_title(
    "By feed category", fontsize=FONTSIZE_TITLE + 2, fontweight="bold", pad=8
)

# Feed by animal
animal_order = feed_animal.mean().sort_values(ascending=False).index.tolist()
df = prepare_luc_df(feed_animal, animal_order)
colors = {c: ANIMAL_COLORS.get(c, "#999999") for c in df.columns}
plot_stacked_emissions(
    df,
    colors,
    axes[1],
    xlabel=GHG_XLABEL,
    ylabel="",
    panel_label="b",
    x_ticks=GHG_XTICKS,
    x_ticklabels=GHG_XTICKLABELS,
    min_height_for_label=0.3,
)
axes[1].set_title(
    "By animal type", fontsize=FONTSIZE_TITLE + 2, fontweight="bold", pad=8
)

plt.tight_layout()
plt.show()

## Summary statistics

In [ ]:
print("=== Net GHG emissions across GHG prices ===")
for price, val in net_emissions.items():
    print(f"  GHG price {price:>6.0f} USD/tCO2eq -> {val:>7.2f} GtCO2eq")

print(f"\nBaseline (price=0): {net_emissions.iloc[0]:.2f} GtCO2eq")
print(
    f"Max price ({net_emissions.index[-1]:.0f}): {net_emissions.iloc[-1]:.2f} GtCO2eq"
)
print(f"Reduction: {net_emissions.iloc[0] - net_emissions.iloc[-1]:.2f} GtCO2eq")
print(
    f"  ({100 * (1 - net_emissions.iloc[-1] / net_emissions.iloc[0]):.1f}% reduction)"
)